# Week 6 Questions
## Apache Spark Architecture and Data Processing
## Introduction

This notebook contains the solutions to the Week 6 questions. It includes both theoretical explanations and practical implementation using PySpark. Wherever required, the answers are demonstrated using executable Spark code along with the corresponding outputs and observations.

### Initial Setup

Before answering the practical questions, it is necessary to import the required PySpark libraries, create a Spark Session, and load the dataset. These steps will be used throughout the notebook.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week6_Mentor_Questions") \
    .getOrCreate()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/source.csv")

df.show(5)

### Ques 1

Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Apache Spark follows a distributed architecture consisting of three major components.

**Driver Program**
The Driver Program is the main component of every Spark application. It creates the Spark Session, converts the user program into execution plans, schedules tasks, and coordinates the complete execution of the application.

**Cluster Manager**
The Cluster Manager is responsible for allocating resources across the cluster. It manages worker nodes and launches executors whenever required.

**Executors**
Executors are worker processes responsible for performing computations on the data. They execute tasks assigned by the Driver, store intermediate results in memory, and return the final output back to the Driver.

### Ques 2 

How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark does not execute transformations immediately. Instead, it records every transformation and builds a Directed Acyclic Graph (DAG). Execution starts only when an action such as `show()`, `count()`, or `write()` is called.

This approach allows Spark to optimize the execution plan by combining multiple transformations, eliminating unnecessary operations, and minimizing data movement. As a result, Spark performs computations more efficiently, especially when processing large datasets.

In [ ]:
lazy_df = df.filter(col("price") > 500) \
            .select("product_id", "category", "price")

print("Transformation created successfully.")
print("No computation has been executed yet.")

lazy_df.show()

**Observation**

The transformations were executed only when the `show()` action was called. This demonstrates Spark's Lazy Evaluation strategy and its ability to optimize execution using a DAG.

### Ques 3

Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [ ]:
csv_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/source.csv")
csv_df.show(5)

**Observation**

The CSV dataset was successfully loaded into a Spark DataFrame. The first row was treated as the header, and Spark automatically inferred the appropriate data types.

### Ques 4

What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

CSV stores data in a row-based format, whereas Parquet stores data in a columnar format.

| CSV | Parquet |
|------|----------|
| Row-based storage | Columnar storage |
| Human-readable | Binary format |
| Larger storage size | Better compression |
| Slower analytical queries | Faster analytical queries |
| Does not store schema | Stores schema information |

Parquet provides better performance because Spark reads only the required columns instead of scanning the complete dataset. This reduces disk I/O and memory usage, making Parquet the preferred format for large-scale data processing.

**Observation**

Parquet offers better storage efficiency and query performance compared to CSV, making it more suitable for big data applications.

### Ques 5

Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the category is 'Electronics'.

In [ ]:
electronics_df = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)
electronics_df.show()

**Observation**

The query filtered the dataset for products belonging to the Electronics category and displayed only the required columns (`product_id` and `price`).

### Ques 6

Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In Spark, column names can be modified using the `withColumnRenamed()` function, while the `cast()` function is used to convert a column from one data type to another. In this example, the column `old_name` is renamed to `new_name`, and the `price` column is converted from String to Double.

In [ ]:
df_q6 = df.withColumnRenamed("old_name", "new_name")

df_q6 = df_q6.withColumn(
    "price",
    col("price").cast("double")
)

df_q6.printSchema()

**Observation**

The column was renamed successfully, and the `price` column was converted to the Double data type. Converting numeric columns to the correct data type is important for performing mathematical calculations accurately.

### Ques 7

Explain how Spark uses the `Lineage Graph (DAG)` for Fault Tolerance if a worker node fails?

Spark maintains a Lineage Graph, also known as a Directed Acyclic Graph (DAG), which records all the transformations applied to a dataset.

If a worker node or partition is lost due to a failure, Spark does not need to reload the entire dataset. Instead, it uses the DAG to recompute only the lost partition from the original data source. This approach provides fault tolerance without requiring data replication after every transformation.

### Ques 8

Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [ ]:
completed_orders = df.filter(
    (col("status") == "Completed") & 
    (col("amount") > 1000)
)

completed_orders.show()

**Observation**

The query returned only those orders that satisfied both conditions. Multiple filtering conditions can be combined using logical operators such as AND (`&`) in Spark.

### Ques 9

Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is a performance optimization used by Spark when working with columnar file formats such as Parquet.

Instead of reading the complete dataset into memory, Spark pushes the filtering condition to the storage layer. As a result, only the required rows and columns are read from disk.

This optimization reduces disk I/O, lowers memory consumption, and significantly improves query performance when working with large datasets.

In [ ]:
filtered_parquet = spark.read.parquet("../output/processed_parquet") \
    .filter(col("category") == "Electronics")

filtered_parquet.show(5)

**Observation**

Only the required records were loaded from the Parquet dataset. In large datasets, Predicate Pushdown helps improve performance by avoiding unnecessary data reads.

### Ques 10

Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

The following code creates a new column named `final_price` by multiplying the `base_price` column by **1.18**, representing an additional 18% tax.

In [ ]:
df_q10 = df.withColumn(
    "final_price",
    round(col("base_price") * 1.18, 2)
)

df_q10.select(
    "base_price",
    "final_price"
).show()

**Observation**

The `final_price` column was generated successfully using the existing `base_price` values. Spark performed the calculation efficiently for every record in the DataFrame.

### Ques 11

What is the difference between Transformations and Actions? Provide two examples of each.

Spark operations are broadly classified into **Transformations** and **Actions**.

### Transformations
Transformations create a new DataFrame from an existing one but do not execute immediately. Instead, Spark records these operations and executes them only when an action is called. This behavior is known as **Lazy Evaluation**.

**Examples:**
- `filter()`
- `select()`

### Actions
Actions trigger the execution of all pending transformations and produce a result or write data to storage.

**Examples:**
- `show()`
- `count()`

In [ ]:
transformation_df = df.filter(col("price") > 500).select(
    "product_id", "price")

print("Transformation created successfully.")

In [ ]:
transformation_df.show()

print("Total Records :", transformation_df.count())

**Observation**

The transformation did not execute immediately. Spark executed the operations only after the `show()` and `count()` actions were called, demonstrating the concept of Lazy Evaluation.

### Ques 12

Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

The following code reads a Parquet dataset, removes rows where the `user_id` column contains null values, and writes the cleaned data into CSV format.

In [ ]:
parquet_df = spark.read.parquet("../output/processed_parquet")

filtered_df = parquet_df.filter(
    col("user_id").isNotNull()
)

filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../output/q12_output_csv")

In [ ]:
filtered_df.show(5)

**Observation**

The Parquet file was loaded successfully. Rows containing null values in the `user_id` column were removed, and the cleaned dataset was saved in CSV format.

### Ques 13

In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

Spark supports two execution modes: **Client Mode** and **Cluster Mode**.

### Client Mode

- The Driver Program runs on the local machine.
- Executors run on worker nodes.
- Suitable for development, testing, and debugging.

### Cluster Mode

- Both the Driver Program and Executors run inside the cluster.
- Better fault tolerance and scalability.
- Preferred for production environments where large datasets are processed.

In this assignment, Spark is running in **Local Mode**, which is commonly used for learning and development purposes.

In [ ]:
print("Application Name :", spark.sparkContext.appName)
print("Spark Version :", spark.version)
print("Master :", spark.sparkContext.master)

**Observation**

The Spark application information was displayed successfully. Since the application is running locally, it is suitable for development and testing.

### Ques 14

Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

The following query filters the dataset to display records where the region is **North** or the priority is **High**.

In [ ]:
north_high = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

north_high.show()

**Observation**

The query returned all records that satisfied either of the given conditions. Logical operators such as OR (`|`) are commonly used to combine multiple filtering conditions in Spark.

### Ques 15

When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

The `show()` function displays only a small number of rows from the DataFrame, making it useful for inspecting data without consuming excessive memory.

On the other hand, the `collect()` function transfers the entire dataset from all executors to the Driver Program. For very large datasets, this can consume a significant amount of memory and may even cause the Driver application to crash due to an OutOfMemory error.

Therefore, when working with large datasets, `show()` is the safer and more efficient option for data exploration.

In [ ]:
df.show(5)

**Observation**

Only the first five records were displayed. Using `show()` allows quick inspection of the dataset while avoiding unnecessary memory usage on the Driver.

# Conclusion

In this notebook, I answered all the Week 6 mentor questions by combining theoretical concepts with practical PySpark implementation. I explored Spark Architecture, Driver, Cluster Manager, Executors, Lazy Evaluation, DAG, Transformations, Actions, Predicate Pushdown, DataFrame operations, and file handling using CSV and Parquet.

The practical examples demonstrated filtering, schema handling, column manipulation, data type conversion, null value handling, and data export. These exercises provided a better understanding of how Spark processes large datasets efficiently and why it is widely used in modern data engineering workflows.